<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/AOFE_V13_5_ControlledScaling_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# CELL 01 — IMPORTS + CONFIG
# ============================================

import json
import math
import numpy as np
from collections import deque

DATA_PATH = "/content/arc-agi_training_challenges.json"

print("CELL 01 loaded.")

CELL 01 loaded.


In [2]:
# ============================================
# CELL 02 — LOAD DATASET + CONVERT FORMAT
# ============================================

def convert_to_solver_format(tasks):
    solver_tasks = {}

    for task_id, task_data in tasks.items():
        if isinstance(task_data, dict) and "train" in task_data and "test" in task_data:
            solver_tasks[task_id] = {
                "train": [
                    {
                        "input": [row[:] for row in pair["input"]],
                        "output": [row[:] for row in pair["output"]],
                    }
                    for pair in task_data["train"]
                ],
                "test": [
                    {
                        "input": [row[:] for row in pair["input"]],
                    }
                    for pair in task_data["test"]
                ],
            }
        elif isinstance(task_data, list):
            solver_tasks[task_id] = {
                "train": [
                    {
                        "input": [row[:] for row in pair["input"]],
                        "output": [row[:] for row in pair["output"]],
                    }
                    for pair in task_data
                    if "input" in pair and "output" in pair
                ],
                "test": [],
            }
        else:
            raise ValueError(f"Unsupported task format for {task_id}")

    return solver_tasks


with open(DATA_PATH, "r") as f:
    raw_tasks = json.load(f)

solver_tasks = convert_to_solver_format(raw_tasks)

sample_id = list(solver_tasks.keys())[0]
sample_task = solver_tasks[sample_id]

print("Loaded raw tasks:", len(raw_tasks))
print("Converted tasks:", len(solver_tasks))
print("Sample task id:", sample_id)
print("Sample keys:", sample_task.keys())
print("Train pairs:", len(sample_task["train"]))
print("Test pairs:", len(sample_task["test"]))
print("CELL 02 loaded.")

Loaded raw tasks: 1000
Converted tasks: 1000
Sample task id: 00576224
Sample keys: dict_keys(['train', 'test'])
Train pairs: 2
Test pairs: 1
CELL 02 loaded.


In [3]:
# ============================================
# CELL 03 — GRID HELPERS
# ============================================

def copy_grid(grid):
    return [row[:] for row in grid]

def to_np(grid):
    return np.array(grid, dtype=int)

def to_list(grid):
    if isinstance(grid, np.ndarray):
        return grid.astype(int).tolist()
    return [list(map(int, row)) for row in grid]

def grid_shape(grid):
    return (len(grid), len(grid[0]) if grid else 0)

def same_shape(a, b):
    return len(a) == len(b) and len(a[0]) == len(b[0])

def unique_colors(grid):
    return sorted({v for row in grid for v in row})

print("CELL 03 loaded.")

CELL 03 loaded.


In [4]:
# ============================================
# CELL 04 — OBJECT EXTRACTION
# ============================================

def get_neighbors(r, c, h, w, connectivity=4):
    nbrs = [(r-1,c), (r+1,c), (r,c-1), (r,c+1)]
    if connectivity == 8:
        nbrs += [(r-1,c-1), (r-1,c+1), (r+1,c-1), (r+1,c+1)]
    return [(nr, nc) for nr, nc in nbrs if 0 <= nr < h and 0 <= nc < w]

def extract_objects(grid, background=0, connectivity=4):
    g = to_np(grid)
    h, w = g.shape
    visited = np.zeros((h, w), dtype=bool)
    objects = []

    for r in range(h):
        for c in range(w):
            if visited[r, c] or g[r, c] == background:
                continue

            color = int(g[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []

            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))

                for nr, nc in get_neighbors(cr, cc, h, w, connectivity):
                    if not visited[nr, nc] and g[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))

            rows = [p[0] for p in pixels]
            cols = [p[1] for p in pixels]

            obj = {
                "id": len(objects),
                "color": color,
                "pixels": pixels,
                "area": len(pixels),
                "bbox": (min(rows), min(cols), max(rows), max(cols)),
                "centroid": (sum(rows) / len(rows), sum(cols) / len(cols)),
                "touches_border": any(
                    pr == 0 or pc == 0 or pr == h - 1 or pc == w - 1
                    for pr, pc in pixels
                ),
            }
            objects.append(obj)

    return objects

print("CELL 04 loaded.")

CELL 04 loaded.


In [5]:
# ============================================
# CELL 05 — ROLE INFERENCE
# ============================================

def infer_object_roles_v2(objects, grid):
    if not objects:
        return {
            "largest": None,
            "smallest": None,
            "topmost": None,
            "bottommost": None,
            "leftmost": None,
            "rightmost": None,
            "center_most": None,
            "border_objects": [],
        }

    h, w = grid_shape(grid)
    center = ((h - 1) / 2.0, (w - 1) / 2.0)

    def center_dist(obj):
        cr, cc = obj["centroid"]
        return (cr - center[0]) ** 2 + (cc - center[1]) ** 2

    largest = max(objects, key=lambda o: (o["area"], -o["id"]))
    smallest = min(objects, key=lambda o: (o["area"], o["id"]))
    topmost = min(objects, key=lambda o: (o["bbox"][0], o["bbox"][1], o["id"]))
    bottommost = max(objects, key=lambda o: (o["bbox"][2], -o["bbox"][1], -o["id"]))
    leftmost = min(objects, key=lambda o: (o["bbox"][1], o["bbox"][0], o["id"]))
    rightmost = max(objects, key=lambda o: (o["bbox"][3], -o["bbox"][0], -o["id"]))
    center_most = min(objects, key=lambda o: (center_dist(o), o["id"]))
    border_objects = [o for o in objects if o["touches_border"]]

    return {
        "largest": largest,
        "smallest": smallest,
        "topmost": topmost,
        "bottommost": bottommost,
        "leftmost": leftmost,
        "rightmost": rightmost,
        "center_most": center_most,
        "border_objects": border_objects,
    }

print("CELL 05 loaded.")

CELL 05 loaded.


In [6]:
# ============================================
# CELL 06 — CORE OPERATORS
# ============================================

def rotate90(grid):
    return np.rot90(to_np(grid), 1)

def rotate180(grid):
    return np.rot90(to_np(grid), 2)

def rotate270(grid):
    return np.rot90(to_np(grid), 3)

def reflect_h(grid):
    return np.flipud(to_np(grid))

def reflect_v(grid):
    return np.fliplr(to_np(grid))

def color_map(grid, mapping):
    g = to_np(grid)
    out = g.copy()
    for src, dst in mapping.items():
        out[g == src] = dst
    return out

def conditional_color_map(grid, source, target):
    g = to_np(grid)
    out = g.copy()
    out[g == source] = target
    return out

def tile_full(grid, nx, ny):
    return np.tile(to_np(grid), (nx, ny))

def crop_to_bbox(grid):
    g = to_np(grid)
    rows = np.any(g != 0, axis=1)
    cols = np.any(g != 0, axis=0)

    if not np.any(rows) or not np.any(cols):
        return g.copy()

    r_idx = np.where(rows)[0]
    c_idx = np.where(cols)[0]
    return g[r_idx[0]:r_idx[-1]+1, c_idx[0]:c_idx[-1]+1]

def extract_largest_object(grid):
    g = to_np(grid)
    objs = extract_objects(g.tolist())
    if not objs:
        return g.copy()

    largest = max(objs, key=lambda o: (o["area"], -o["id"]))
    out = np.zeros_like(g)
    for r, c in largest["pixels"]:
        out[r, c] = g[r, c]
    return out

def recolor_role(grid, role, color):
    g = to_np(grid)
    objs = extract_objects(g.tolist())
    roles = infer_object_roles_v2(objs, g.tolist())

    if role not in roles or roles[role] is None:
        return g.copy()

    out = g.copy()
    for r, c in roles[role]["pixels"]:
        out[r, c] = color
    return out

def translate_role(grid, role, dx, dy, background=0):
    g = to_np(grid)
    objs = extract_objects(g.tolist(), background=background)
    roles = infer_object_roles_v2(objs, g.tolist())

    if role not in roles or roles[role] is None:
        return g.copy()

    obj = roles[role]
    out = g.copy()

    for r, c in obj["pixels"]:
        out[r, c] = background

    for r, c in obj["pixels"]:
        nr = r + dy
        nc = c + dx
        if 0 <= nr < g.shape[0] and 0 <= nc < g.shape[1]:
            out[nr, nc] = obj["color"]

    return out

print("CELL 06 loaded.")

CELL 06 loaded.


In [7]:
# ============================================
# CELL 07 — OPERATOR REGISTRY (UNIFIED + TASK-FILTERED)
# ============================================

def detect_task_features(task):
    pair = task["train"][0]
    inp = pair["input"]
    out = pair["output"]

    in_h, in_w = len(inp), len(inp[0])
    out_h, out_w = len(out), len(out[0])

    same_shape_flag = (in_h == out_h and in_w == out_w)
    expansion_flag = (out_h > in_h or out_w > in_w)

    color_change_flag = False
    overlap_h = min(in_h, out_h)
    overlap_w = min(in_w, out_w)

    for r in range(overlap_h):
        for c in range(overlap_w):
            if inp[r][c] != out[r][c]:
                color_change_flag = True
                break
        if color_change_flag:
            break

    return {
        "same_shape": same_shape_flag,
        "expansion": expansion_flag,
        "color_change": color_change_flag,
    }


def get_all_ops(task):
    out_colors = sorted({
        v
        for pair in task["train"]
        for row in pair["output"]
        for v in row
    })

    roles = [
        "largest",
        "smallest",
        "center_most",
        "topmost",
        "bottommost",
        "leftmost",
        "rightmost",
    ]

    ops = []

    # ----------------------------------
    # GEOMETRIC / STRUCTURAL
    # ----------------------------------
    ops.extend([
        ("rotate90", {}),
        ("rotate180", {}),
        ("rotate270", {}),
        ("reflect_h", {}),
        ("reflect_v", {}),
        ("tile_full", {"nx": 2, "ny": 2}),
        ("tile_full", {"nx": 3, "ny": 3}),
        ("crop_to_bbox", {}),
        ("extract_largest_object", {}),
        ("crop_nonzero_bbox", {}),
        ("isolate_objects", {}),
        ("compress_vertical", {}),
        ("compress_horizontal", {}),
    ])

    # ----------------------------------
    # COLOR OPS
    # ----------------------------------
    for s in range(10):
        for t in range(10):
            if s != t:
                ops.append(("conditional_color_map", {"source": s, "target": t}))

    # inferred global color map from first train pair
    pair0 = task["train"][0]
    inferred_map = {}
    consistent = True

    inp = pair0["input"]
    out = pair0["output"]

    if same_shape(inp, out):
        for r in range(len(inp)):
            for c in range(len(inp[0])):
                a = inp[r][c]
                b = out[r][c]

                if a in inferred_map and inferred_map[a] != b:
                    consistent = False
                    break
                inferred_map[a] = b

            if not consistent:
                break

    if consistent and inferred_map:
        ops.append(("color_map", {"mapping": inferred_map}))

    # ----------------------------------
    # ROLE OPS
    # ----------------------------------
    for role in roles:
        for color in out_colors[:8]:
            ops.append(("recolor_role", {"role": role, "color": color}))

        ops.append(("translate_role", {"role": role, "dx": 1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": -1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": 1}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": -1}))

    return ops


def build_task_ops(task):
    all_ops = get_all_ops(task)
    features = detect_task_features(task)

    filtered = []

    for op in all_ops:
        name = op[0]

        # ----------------------------------
        # EXPANSION / TILING TASKS
        # ----------------------------------
        if features["expansion"]:
            if name in [
                "tile_full",
                "crop_nonzero_bbox",
                "crop_to_bbox",
                "compress_vertical",
                "compress_horizontal",
                "isolate_objects",
                "extract_largest_object",
                "reflect_h",
                "reflect_v",
                "rotate90",
                "rotate180",
                "rotate270",
            ]:
                filtered.append(op)

        # ----------------------------------
        # SAME-SHAPE COLOR / OBJECT TASKS
        # ----------------------------------
        elif features["same_shape"] and features["color_change"]:
            if name in [
                "color_map",
                "conditional_color_map",
                "recolor_role",
                "translate_role",
                "extract_largest_object",
                "crop_nonzero_bbox",
            ]:
                filtered.append(op)

        # ----------------------------------
        # DEFAULT
        # ----------------------------------
        else:
            filtered.append(op)

    if not filtered:
        filtered = all_ops

    return filtered


# quick sanity check on first task
_first_task_id = list(solver_tasks.keys())[0]
_first_task = solver_tasks[_first_task_id]
_first_ops = build_task_ops(_first_task)

print("CELL 07 loaded — unified task-filtered operator registry active.")
print("First task id:", _first_task_id)
print("First task op count:", len(_first_ops))
print("First task first 20 ops:", _first_ops[:20])

CELL 07 loaded — unified task-filtered operator registry active.
First task id: 00576224
First task op count: 13
First task first 20 ops: [('rotate90', {}), ('rotate180', {}), ('rotate270', {}), ('reflect_h', {}), ('reflect_v', {}), ('tile_full', {'nx': 2, 'ny': 2}), ('tile_full', {'nx': 3, 'ny': 3}), ('crop_to_bbox', {}), ('extract_largest_object', {}), ('crop_nonzero_bbox', {}), ('isolate_objects', {}), ('compress_vertical', {}), ('compress_horizontal', {})]


In [8]:
# ============================================
# CELL 7b — FINALIZATION OPERATORS
# ============================================

import numpy as np

# ----------------------------------
# CROP NON-ZERO BOUNDING BOX
# ----------------------------------
def crop_nonzero_bbox(grid):
    g = np.array(grid)
    rows = np.any(g != 0, axis=1)
    cols = np.any(g != 0, axis=0)

    if not np.any(rows) or not np.any(cols):
        return g

    r = np.where(rows)[0]
    c = np.where(cols)[0]

    return g[r[0]:r[-1]+1, c[0]:c[-1]+1]


# ----------------------------------
# REMOVE BACKGROUND (SET TO 0 OUTSIDE OBJECTS)
# ----------------------------------
def isolate_objects(grid):
    g = np.array(grid)
    objs = extract_objects(g.tolist())

    out = np.zeros_like(g)

    for obj in objs:
        for r, c in obj["pixels"]:
            out[r, c] = g[r, c]

    return out


# ----------------------------------
# STACK OBJECTS (VERTICAL COMPRESSION)
# ----------------------------------
def compress_vertical(grid):
    g = np.array(grid)
    rows = [row for row in g if np.any(row != 0)]

    if not rows:
        return g

    return np.array(rows)


# ----------------------------------
# STACK OBJECTS (HORIZONTAL COMPRESSION)
# ----------------------------------
def compress_horizontal(grid):
    g = np.array(grid)
    cols = [g[:, i] for i in range(g.shape[1]) if np.any(g[:, i] != 0)]

    if not cols:
        return g

    return np.stack(cols, axis=1)


print("CELL 7b loaded — finalization operators ready.")

CELL 7b loaded — finalization operators ready.


In [9]:
# ============================================
# CELL 08 — PROGRAM EXECUTION (FULL)
# ============================================

def run_program(program, grid):

    current = to_np(grid)

    for op_name, params in program:

        try:
            # ----------------------------------
            # TRANSFORM OPS
            # ----------------------------------
            if op_name == "rotate90":
                current = rotate90(current)

            elif op_name == "rotate180":
                current = rotate180(current)

            elif op_name == "rotate270":
                current = rotate270(current)

            elif op_name == "reflect_h":
                current = reflect_h(current)

            elif op_name == "reflect_v":
                current = reflect_v(current)

            elif op_name == "tile_full":
                current = tile_full(current, params["nx"], params["ny"])

            # ----------------------------------
            # COLOR OPS
            # ----------------------------------
            elif op_name == "color_map":
                current = color_map(current, params["mapping"])

            elif op_name == "conditional_color_map":
                current = conditional_color_map(current, params["source"], params["target"])

            # ----------------------------------
            # STRUCTURE OPS
            # ----------------------------------
            elif op_name == "crop_to_bbox":
                current = crop_to_bbox(current)

            elif op_name == "extract_largest_object":
                current = extract_largest_object(current)

            # ----------------------------------
            # ROLE OPS
            # ----------------------------------
            elif op_name == "recolor_role":
                current = recolor_role(current, params["role"], params["color"])

            elif op_name == "translate_role":
                current = translate_role(current, params["role"], params["dx"], params["dy"])

            # ----------------------------------
            # 🔥 FINALIZATION OPS (NEW)
            # ----------------------------------
            elif op_name == "crop_nonzero_bbox":
                current = crop_nonzero_bbox(current)

            elif op_name == "isolate_objects":
                current = isolate_objects(current)

            elif op_name == "compress_vertical":
                current = compress_vertical(current)

            elif op_name == "compress_horizontal":
                current = compress_horizontal(current)

            # ----------------------------------
            # UNKNOWN OP SAFETY
            # ----------------------------------
            else:
                return None

        except Exception:
            return None

    return to_list(current)


print("CELL 08 loaded — execution engine ready.")

CELL 08 loaded — execution engine ready.


In [10]:
# ============================================
# CELL 09 — AOFE ERROR ANALYSIS + SCORING
# ============================================

def analyze_errors(pred, target):
    pred = to_list(pred)
    target = to_list(target)

    same = same_shape(pred, target)

    h = max(len(pred), len(target))
    w = max(len(pred[0]), len(target[0]))

    error_map = []
    error_count = 0

    for r in range(h):
        row = []
        for c in range(w):
            pv = pred[r][c] if r < len(pred) and c < len(pred[0]) else None
            tv = target[r][c] if r < len(target) and c < len(target[0]) else None
            mismatch = int(pv != tv)
            row.append(mismatch)
            error_count += mismatch
        error_map.append(row)

    total = len(target) * len(target[0]) if same else h * w
    error_density = error_count / max(1, total)
    pixel_accuracy = 1.0 - (error_count / max(1, total)) if same else 0.0

    pred_colors = set(v for row in pred for v in row)
    tgt_colors = set(v for row in target for v in row)
    color_score = len(pred_colors & tgt_colors) / max(1, len(pred_colors | tgt_colors))

    try:
        pred_objs = extract_objects(pred)
        tgt_objs = extract_objects(target)
        struct_count = 1.0 - abs(len(pred_objs) - len(tgt_objs)) / max(1, max(len(pred_objs), len(tgt_objs)))
    except Exception:
        struct_count = 0.0

    structural_score = 0.5 * float(same) + 0.5 * struct_count

    return {
        "error_map": error_map,
        "error_count": error_count,
        "error_density": max(0.0, min(1.0, error_density)),
        "pixel_accuracy": max(0.0, min(1.0, pixel_accuracy)),
        "color_score": max(0.0, min(1.0, color_score)),
        "structural_score": max(0.0, min(1.0, structural_score)),
    }

def calculate_aofe_score(pred, target):
    a = analyze_errors(pred, target)
    score = (
        0.60 * a["pixel_accuracy"] +
        0.25 * a["structural_score"] +
        0.15 * a["color_score"]
    )
    return max(0.0, min(1.0, score))

def eval_program_with_errors(program, task):
    analyses = []
    pair_scores = []

    for pair in task["train"]:
        pred = run_program(program, pair["input"])
        if pred is None:
            return None

        analysis = analyze_errors(pred, pair["output"])
        score = calculate_aofe_score(pred, pair["output"])

        analyses.append(analysis)
        pair_scores.append(score)

    return {
        "score": sum(pair_scores) / max(1, len(pair_scores)),
        "analyses": analyses,
        "pair_scores": pair_scores,
    }

print("CELL 09 loaded.")

CELL 09 loaded.


In [38]:
# ============================================
# CELL 10 — PRIORITY SYSTEM (STRUCTURE-BOOST V2)
# ============================================

# Version tracking
SYSTEM_STATE["priority_version"] = "structure_boost_v2"

def get_priority_ops(eval_result, task):
    """
    Advanced priority system:
    - Promotes structural reasoning
    - Activates color logic
    - Controls translation dominance
    - Uses failure-aware boosting
    """

    failure = eval_result["failure_signature"]["type"]
    pair_errors = eval_result["error_analyses"]

    # Get task-aware operator pool
    task_ops = build_task_ops(task)

    scored_ops = []

    for op in task_ops:
        score = 0
        op_name = op["op"]
        params = op.get("params", {})

        # ----------------------------------
        # 🔥 STRUCTURE BOOST (MOST IMPORTANT)
        # ----------------------------------
        if op_name in ["crop_to_bbox", "extract_largest_object"]:
            score += 15

        # ----------------------------------
        # 🎨 COLOR LOGIC BOOST
        # ----------------------------------
        if op_name in ["color_map", "conditional_color_map"]:
            score += 10

        # ----------------------------------
        # 🔄 TRANSFORM PRIORITY
        # ----------------------------------
        if op_name in [
            "rotate90", "rotate180", "rotate270",
            "reflect_h", "reflect_v", "transpose"
        ]:
            score += 8

        # ----------------------------------
        # 📦 OBJECT / ROLE OPS (if present)
        # ----------------------------------
        if op_name in ["recolor_role", "translate_role"]:
            score += 9

        # ----------------------------------
        # 🚫 CONTROLLED TRANSLATION (NERFED)
        # ----------------------------------
        if op_name == "translate":
            dx = abs(int(params.get("dx", 0)))
            dy = abs(int(params.get("dy", 0)))
            score += max(0, 2 - dx - dy)   # heavily reduced influence

        # ----------------------------------
        # 🧠 FAILURE-AWARE BOOSTING
        # ----------------------------------
        if failure == "shape_mismatch":
            if op_name in ["tile_n", "crop_to_bbox"]:
                score += 10

        elif failure == "object_placement_mismatch":
            if op_name == "translate":
                score += 6

        elif failure == "near_miss_or_missing_operator":
            if op_name in ["conditional_color_map", "extract_largest_object"]:
                score += 12

        elif failure == "content_mismatch":
            if op_name in ["color_map", "conditional_color_map"]:
                score += 8

        # ----------------------------------
        # 📊 DENSITY SIGNAL (fine-tuning)
        # ----------------------------------
        if pair_errors:
            avg_density = sum(e["error_density"] for e in pair_errors) / len(pair_errors)

            if avg_density > 0.4:
                if op_name in ["crop_to_bbox", "extract_largest_object"]:
                    score += 5

            elif avg_density < 0.15:
                if op_name in ["color_map", "conditional_color_map"]:
                    score += 5

        # ----------------------------------
        # 💀 KILL IDENTITY (VERY IMPORTANT)
        # ----------------------------------
        if op_name == "identity":
            score -= 100

        scored_ops.append((score, op))

    # ----------------------------------
    # SORT BY SCORE
    # ----------------------------------
    scored_ops.sort(key=lambda x: x[0], reverse=True)

    # ----------------------------------
    # RETURN ORDERED OPS
    # ----------------------------------
    return [op for _, op in scored_ops]


print("CELL 10 loaded — PRIORITY SYSTEM V2 ACTIVE.")

CELL 10 loaded — PRIORITY SYSTEM V2 ACTIVE.


In [39]:
# ============================================
# CELL 10b — ARCHETYPE-AWARE PRIORITY OPS
# ============================================

def detect_task_archetype(task):
    pair = task["train"][0]
    inp = pair["input"]
    out = pair["output"]

    in_h, in_w = len(inp), len(inp[0])
    out_h, out_w = len(out), len(out[0])

    # shape expansion / tiling clue
    if out_h > in_h or out_w > in_w:
        return "tiling_or_expansion"

    # color-change clue
    if (in_h == out_h) and (in_w == out_w):
        changed = 0
        same = 0
        for r in range(in_h):
            for c in range(in_w):
                if inp[r][c] == out[r][c]:
                    same += 1
                else:
                    changed += 1

        if changed > 0 and same > 0:
            return "localized_recolor"

    return "generic"


def get_priority_ops(analysis, ops, task=None):
    error_density = analysis["error_density"]
    pixel_accuracy = analysis["pixel_accuracy"]
    structural_score = analysis["structural_score"]

    transform_ops = []
    color_ops = []
    object_ops = []
    finalization_ops = []
    extraction_ops = []
    other_ops = []

    for op in ops:
        name = op[0]

        if name in ["rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "tile_full"]:
            transform_ops.append(op)
        elif name in ["conditional_color_map", "color_map"]:
            color_ops.append(op)
        elif name in ["recolor_role", "translate_role"]:
            object_ops.append(op)
        elif name in ["crop_nonzero_bbox", "compress_vertical", "compress_horizontal", "isolate_objects"]:
            finalization_ops.append(op)
        elif name in ["crop_to_bbox", "extract_largest_object"]:
            extraction_ops.append(op)
        else:
            other_ops.append(op)

    archetype = detect_task_archetype(task) if task is not None else "generic"

    # ----------------------------------
    # TILING / EXPANSION TASKS
    # ----------------------------------
    if archetype == "tiling_or_expansion":
        ordered = (
            transform_ops +
            finalization_ops +
            extraction_ops +
            object_ops +
            color_ops +
            other_ops
        )

    # ----------------------------------
    # STRUCTURE GOOD, PIXELS WRONG
    # ----------------------------------
    elif structural_score > 0.8 and pixel_accuracy < 1.0:
        ordered = (
            color_ops +
            object_ops +
            finalization_ops +
            transform_ops +
            extraction_ops +
            other_ops
        )

    # ----------------------------------
    # HIGH ERROR
    # ----------------------------------
    elif error_density > 0.25:
        ordered = (
            transform_ops +
            extraction_ops +
            object_ops +
            finalization_ops +
            color_ops +
            other_ops
        )

    # ----------------------------------
    # DEFAULT
    # ----------------------------------
    else:
        ordered = (
            object_ops +
            color_ops +
            finalization_ops +
            transform_ops +
            extraction_ops +
            other_ops
        )

    return ordered


print("CELL 10b loaded — archetype-aware priority ops active.")

CELL 10b loaded — archetype-aware priority ops active.


In [40]:
# ============================================
# CELL 11 — AOFE BEAM SEARCH
# ============================================

def normalize_program(program):
    return [op for op in program]

def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    if isinstance(obj, list):
        return tuple(make_hashable(x) for x in obj)
    return obj

def program_signature(program):
    return tuple((name, make_hashable(params)) for name, params in normalize_program(program))

def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores, best_analyses, best_refinement_steps

        p = normalize_program(program)
        sig = program_signature(p)

        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(p, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        mdl_penalty = 0.001 * len(p)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (raw_score == best_score and (best_program is None or len(p) < len(best_program))):
            best_program = p
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(p), p, pair_scores, analyses, refinement_steps)

    initial_programs = [[]]
    for op in ops:
        initial_programs.append([op])

    ranked = []
    for prog in initial_programs:
        item = consider(prog, refinement_steps=0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):
        candidates = []

        for _, _, program, _, analyses, _ in ranked:
            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops)
            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:
                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, refinement_steps=step + 1)
                if item is not None:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score >= 0.999:
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }

print("CELL 11 loaded.")

CELL 11 loaded.


In [41]:
# ============================================
# CELL 11b — SOLUTION LOCK + EARLY STOP FIX
# ============================================

def is_perfect_solution(result):
    if result is None:
        return False

    # strict check
    return all(a["error_density"] == 0.0 for a in result["analyses"])


def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    PERFECT_SOLUTION = None

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores
        nonlocal best_analyses, best_refinement_steps, PERFECT_SOLUTION

        sig = program_signature(program)
        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(program, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        # 🚨 SOLUTION LOCK
        if is_perfect_solution(result):
            PERFECT_SOLUTION = {
                "program": program,
                "score": raw_score,
                "analyses": analyses,
                "pair_scores": pair_scores,
                "steps": refinement_steps
            }

        mdl_penalty = 0.001 * len(program)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (
            raw_score == best_score and (
                best_program is None or len(program) < len(best_program)
            )
        ):
            best_program = program
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(program), program, pair_scores, analyses, refinement_steps)

    # initial programs
    initial_programs = [[]] + [[op] for op in ops]

    ranked = []
    for prog in initial_programs:
        item = consider(prog, 0)
        if item:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):

        # 🚨 EARLY STOP IF PERFECT FOUND
        if PERFECT_SOLUTION is not None:
            break

        candidates = []

        for _, _, program, _, analyses, _ in ranked:

            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops)

            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:

                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, step + 1)

                if item:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

    # 🚨 RETURN PERFECT IF FOUND
    if PERFECT_SOLUTION is not None:
        return {
            "best_program": PERFECT_SOLUTION["program"],
            "best_score": 1.0,
            "pair_scores": PERFECT_SOLUTION["pair_scores"],
            "refinement_steps": PERFECT_SOLUTION["steps"],
            "error_density_progression": [0.0],
            "failure_mode": None,
            "ranked": ranked,
        }

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }

print("CELL 11b loaded — solution lock active.")

CELL 11b loaded — solution lock active.


In [42]:
# ============================================
# VERIFY ACTIVE beam_search VERSION
# ============================================

import inspect

src = inspect.getsource(beam_search)

print("---- BEAM SEARCH SOURCE (FIRST 300 CHARS) ----")
print(src[:300])

print("\nContains solution lock?:", "PERFECT_SOLUTION" in src)
print("Contains early stop?:", "EARLY STOP" in src or "PERFECT_SOLUTION" in src)

---- BEAM SEARCH SOURCE (FIRST 300 CHARS) ----
def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    PERFECT_SOLUTION = None

    def consider(program, refinement_steps):
        n

Contains solution lock?: True
Contains early stop?: True


In [43]:
# ============================================
# CELL 11d — COMPOSITION SEED TEMPLATES
# ============================================

def seed_programs_from_task(task):
    ops = build_task_ops(task)
    features = detect_task_features(task)

    seeds = []

    # ----------------------------------
    # Always allow empty start
    # ----------------------------------
    seeds.append([])

    # ----------------------------------
    # Single-op seeds
    # ----------------------------------
    for op in ops:
        seeds.append([op])

    # ----------------------------------
    # Expansion / tiling task seeds
    # ----------------------------------
    if features["expansion"]:
        tile_ops = [op for op in ops if op[0] == "tile_full"]
        finalize_ops = [op for op in ops if op[0] in [
            "crop_nonzero_bbox",
            "crop_to_bbox",
            "compress_vertical",
            "compress_horizontal",
            "isolate_objects",
            "extract_largest_object",
        ]]

        geom_ops = [op for op in ops if op[0] in [
            "reflect_h", "reflect_v", "rotate90", "rotate180", "rotate270"
        ]]

        # tile -> finalize
        for t in tile_ops:
            for f in finalize_ops:
                seeds.append([t, f])

        # extract -> tile
        for f in finalize_ops:
            for t in tile_ops:
                seeds.append([f, t])

        # geom -> tile
        for g in geom_ops:
            for t in tile_ops:
                seeds.append([g, t])

        # geom -> tile -> finalize
        for g in geom_ops:
            for t in tile_ops:
                for f in finalize_ops[:3]:
                    seeds.append([g, t, f])

    # ----------------------------------
    # Same-shape color/object task seeds
    # ----------------------------------
    elif features["same_shape"] and features["color_change"]:
        color_ops = [op for op in ops if op[0] in [
            "color_map",
            "conditional_color_map",
        ]]
        role_ops = [op for op in ops if op[0] == "recolor_role"]
        move_ops = [op for op in ops if op[0] == "translate_role"]

        # color -> recolor
        for c in color_ops:
            for r in role_ops[:12]:
                seeds.append([c, r])

        # recolor -> move
        for r in role_ops[:12]:
            for m in move_ops[:8]:
                seeds.append([r, m])

        # color -> recolor -> move
        for c in color_ops[:10]:
            for r in role_ops[:8]:
                for m in move_ops[:4]:
                    seeds.append([c, r, m])

    # ----------------------------------
    # Deduplicate
    # ----------------------------------
    unique = []
    seen = set()

    for prog in seeds:
        sig = program_signature(prog)
        if sig not in seen:
            seen.add(sig)
            unique.append(prog)

    return unique


def is_perfect_solution(result):
    if result is None:
        return False
    return all(a["error_density"] == 0.0 for a in result["analyses"])


def beam_search(task, width=12, depth=4):
    ops = build_task_ops(task)
    seen = set()

    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    perfect_solution = None

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores
        nonlocal best_analyses, best_refinement_steps, perfect_solution

        sig = program_signature(program)
        if sig in seen:
            return None
        seen.add(sig)

        result = eval_program_with_errors(program, task)
        if result is None:
            return None

        raw_score = result["score"]
        analyses = result["analyses"]
        pair_scores = result["pair_scores"]

        if is_perfect_solution(result):
            perfect_solution = {
                "program": program,
                "score": raw_score,
                "analyses": analyses,
                "pair_scores": pair_scores,
                "steps": refinement_steps,
            }

        mdl_penalty = 0.001 * len(program)
        final_score = raw_score - mdl_penalty

        if raw_score > best_score or (
            raw_score == best_score and (
                best_program is None or len(program) < len(best_program)
            )
        ):
            best_program = program
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return (final_score, -len(program), program, pair_scores, analyses, refinement_steps)

    # ----------------------------------
    # Seeded initial programs (NEW)
    # ----------------------------------
    initial_programs = seed_programs_from_task(task)

    ranked = []
    for prog in initial_programs:
        item = consider(prog, 0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    for step in range(depth):
        if perfect_solution is not None:
            break

        candidates = []

        for _, _, program, _, analyses, _ in ranked:
            if not analyses:
                continue

            worst = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst, ops, task=task)
            last_op = program[-1][0] if program else None

            for op in ordered_ops[:width]:
                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, step + 1)

                if item is not None:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

    if perfect_solution is not None:
        return {
            "best_program": perfect_solution["program"],
            "best_score": 1.0,
            "pair_scores": perfect_solution["pair_scores"],
            "refinement_steps": perfect_solution["steps"],
            "error_density_progression": [0.0],
            "failure_mode": None,
            "ranked": ranked,
        }

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [round(a["error_density"], 4) for a in best_analyses],
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator",
        "ranked": ranked,
    }


print("CELL 11d loaded — composition seed templates active.")

CELL 11d loaded — composition seed templates active.


In [44]:
# ============================================
# CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH
# ============================================
# Adds:
# - SYSTEM_STATE / CELL_REGISTRY / RUN_REPORT / OP_USAGE / PAST_RUNS
# - analyze_errors
# - calculate_aofe_score
# - eval_program_with_errors
# - get_priority_ops
# - build_task_ops (safe fallback if missing)
# - beam_search override with density-guided refinement
# - introspection + controlled test harness
# ============================================

import inspect
import math
from copy import deepcopy
from collections import defaultdict

# ============================================
# GLOBAL STATE
# ============================================

SYSTEM_STATE = globals().get("SYSTEM_STATE", {
    "active_cells": [],
    "beam_search_version": None,
    "ops_version": None,
    "priority_version": None,
    "last_run_summary": None,
})

CELL_REGISTRY = globals().get("CELL_REGISTRY", {
    "07": "operator_system",
    "10": "priority_system",
    "11c": "beam_core",
    "11d": "seed_templates",
    "12": "aofe_v13_5_core_patch",
})

RUN_REPORT = globals().get("RUN_REPORT", {
    "avg_score": 0.0,
    "solve_rate": 0.0,
    "task_results": {}
})

OP_USAGE = globals().get("OP_USAGE", defaultdict(int))
PAST_RUNS = globals().get("PAST_RUNS", [])
FILES_INDEX = globals().get("FILES_INDEX", {})

SYSTEM_STATE["active_cells"].append("CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH")

print("AOFE patch cell starting...")

# ============================================
# SAFETY HELPERS
# ============================================

def _safe_to_list(grid):
    if isinstance(grid, np.ndarray):
        return grid.astype(int).tolist()
    return [list(map(int, row)) for row in grid]

def _grid_shape(grid):
    grid = _safe_to_list(grid)
    return (len(grid), len(grid[0]) if grid else 0)

def _same_shape(a, b):
    a = _safe_to_list(a)
    b = _safe_to_list(b)
    if not a or not b:
        return False
    return len(a) == len(b) and len(a[0]) == len(b[0])

def _nonzero_bbox(grid):
    g = np.array(_safe_to_list(grid), dtype=int)
    coords = np.argwhere(g != 0)
    if len(coords) == 0:
        return None
    r0, c0 = coords.min(axis=0)
    r1, c1 = coords.max(axis=0)
    return (int(r0), int(c0), int(r1), int(c1))

def _normalize_program(program):
    out = []
    for step in program:
        if isinstance(step, dict):
            out.append(step)
        elif isinstance(step, tuple) and len(step) == 2 and isinstance(step[1], dict):
            out.append({"op": step[0], "params": step[1]})
        elif isinstance(step, tuple) and len(step) == 1:
            out.append({"op": step[0], "params": {}})
        elif isinstance(step, str):
            out.append({"op": step, "params": {}})
        else:
            out.append({"op": str(step), "params": {}})
    return out

def _make_hashable(x):
    if isinstance(x, dict):
        return tuple(sorted((k, _make_hashable(v)) for k, v in x.items()))
    elif isinstance(x, list):
        return tuple(_make_hashable(v) for v in x)
    elif isinstance(x, tuple):
        return tuple(_make_hashable(v) for v in x)
    elif isinstance(x, np.ndarray):
        return tuple(tuple(int(v) for v in row) for row in x.tolist())
    else:
        return x

def _program_to_key(program):
    return tuple(
        (step["op"], _make_hashable(step.get("params", {})))
        for step in _normalize_program(program)
    )

def _mdl_cost(program):
    cost = 0
    for step in _normalize_program(program):
        cost += 1 + len(step.get("params", {}))
    return cost

def _clip01(x):
    return max(0.0, min(1.0, float(x)))

def _deepcopy_grid(grid):
    return [row[:] for row in _safe_to_list(grid)]

# ============================================
# FILE INDEX
# ============================================

def build_files_index():
    global FILES_INDEX
    candidate_names = [
        "arc-agi_training_challenges (3).json",
        "arc_trace_dataset.json",
        "arc_trace_dataset_v2.json",
        "arc_memory_db.json",
        "arc_compositional_memory_db.json",
        "aofe_v13_5_controlledscaling_lab.py",
        "AOFE_V13_5_ControlledScaling_Lab.ipynb",
        "Pasted text.txt",
        "Pasted text (2).txt",
        "Pasted text (3).txt",
    ]
    FILES_INDEX = {name: True for name in candidate_names}
    return FILES_INDEX

build_files_index()
print("FILES_INDEX entries:", len(FILES_INDEX))

# ============================================
# OP EXECUTION LAYER
# ============================================

def execute_step(grid, step):
    step = step if isinstance(step, dict) else {"op": str(step), "params": {}}
    op = step["op"]
    params = step.get("params", {})

    g = _safe_to_list(grid)

    if op == "identity":
        return _safe_to_list(g)

    elif op == "rotate90":
        return _safe_to_list(np.rot90(np.array(g), 1))

    elif op == "rotate180":
        return _safe_to_list(np.rot90(np.array(g), 2))

    elif op == "rotate270":
        return _safe_to_list(np.rot90(np.array(g), 3))

    elif op == "reflect_h":
        return _safe_to_list(np.flipud(np.array(g)))

    elif op == "reflect_v":
        return _safe_to_list(np.fliplr(np.array(g)))

    elif op == "transpose":
        return _safe_to_list(np.array(g).T)

    elif op == "crop_to_bbox":
        bbox = _nonzero_bbox(g)
        if bbox is None:
            return _safe_to_list(g)
        r0, c0, r1, c1 = bbox
        arr = np.array(g)
        return _safe_to_list(arr[r0:r1+1, c0:c1+1])

    elif op == "tile_n":
        n = int(params.get("n", 1))
        return _safe_to_list(np.tile(np.array(g), (n, n)))

    elif op == "row_alt_tile_n":
        n = int(params.get("n", 1))
        arr = np.array(g)
        blocks = []
        for i in range(n):
            row_blocks = []
            for _ in range(n):
                block = np.fliplr(arr) if (i % 2 == 1) else arr
                row_blocks.append(block)
            blocks.append(np.concatenate(row_blocks, axis=1))
        return _safe_to_list(np.concatenate(blocks, axis=0))

    elif op == "translate":
        dx = int(params.get("dx", 0))
        dy = int(params.get("dy", 0))
        fill = int(params.get("fill", 0))
        arr = np.array(g)
        h, w = arr.shape
        out = np.full((h, w), fill, dtype=int)
        for r in range(h):
            for c in range(w):
                nr, nc = r + dy, c + dx
                if 0 <= nr < h and 0 <= nc < w:
                    out[nr, nc] = arr[r, c]
        return _safe_to_list(out)

    elif op == "color_map":
        mapping = params.get("mapping", params.get("color_map", {}))
        arr = np.array(g)
        out = arr.copy()
        for src, dst in mapping.items():
            out[arr == int(src)] = int(dst)
        return _safe_to_list(out)

    elif op == "conditional_color_map":
        src = int(params.get("source", 0))
        dst = int(params.get("target", 0))
        arr = np.array(g)
        out = arr.copy()
        out[arr == src] = dst
        return _safe_to_list(out)

    elif op == "extract_largest_object":
        objs = extract_objects(g) if "extract_objects" in globals() else []
        if not objs:
            return _safe_to_list(g)
        largest = max(objs, key=lambda o: (o["area"], -o["id"]))
        r0, c0, r1, c1 = largest["bbox"]
        arr = np.array(g)
        out = np.zeros((r1-r0+1, c1-c0+1), dtype=int)
        for pr, pc in largest["pixels"]:
            out[pr-r0, pc-c0] = largest["color"]
        return _safe_to_list(out)

    return _safe_to_list(g)

def run_program_safe(program, grid):
    prog = _normalize_program(program)
    cur = _safe_to_list(grid)
    for step in prog:
        cur = execute_step(cur, step)
    return cur

# Prefer notebook's run_program if it exists and works.
def run_program_compat(program, grid):
    if "run_program" in globals():
        try:
            return _safe_to_list(run_program(program, grid))
        except Exception:
            pass
    return run_program_safe(program, grid)

# ============================================
# ERROR ANALYSIS
# ============================================

def analyze_errors(pred, target):
    pred = _safe_to_list(pred)
    target = _safe_to_list(target)

    ph, pw = _grid_shape(pred)
    th, tw = _grid_shape(target)

    h = max(ph, th)
    w = max(pw, tw)

    error_map = []
    error_count = 0
    row_error_counts = []
    col_error_counts = [0] * w

    for r in range(h):
        row = []
        row_err = 0
        for c in range(w):
            pv = pred[r][c] if r < ph and c < pw else None
            tv = target[r][c] if r < th and c < tw else None
            mismatch = int(pv != tv)
            row.append(mismatch)
            error_count += mismatch
            row_err += mismatch
            col_error_counts[c] += mismatch
        error_map.append(row)
        row_error_counts.append(row_err)

    total_cells = max(1, h * w)
    same_shape_flag = (ph == th and pw == tw)

    pixel_accuracy = 1.0 - (error_count / total_cells)

    pred_colors = set(v for row in pred for v in row) if ph > 0 and pw > 0 else set()
    tgt_colors = set(v for row in target for v in row) if th > 0 and tw > 0 else set()
    union = pred_colors | tgt_colors
    inter = pred_colors & tgt_colors
    color_score = len(inter) / max(1, len(union))

    try:
        pred_objs = extract_objects(pred) if "extract_objects" in globals() else []
        tgt_objs = extract_objects(target) if "extract_objects" in globals() else []
        pred_count = len(pred_objs)
        tgt_count = len(tgt_objs)
        structural_count_score = 1.0 - abs(pred_count - tgt_count) / max(1, max(pred_count, tgt_count))
    except Exception:
        structural_count_score = 0.0

    shape_score = 1.0 if same_shape_flag else 0.0
    structural_score = 0.5 * shape_score + 0.5 * structural_count_score

    bbox_pred = _nonzero_bbox(pred)
    bbox_tgt = _nonzero_bbox(target)
    bbox_mismatch = 0
    if bbox_pred != bbox_tgt:
        bbox_mismatch = 1

    density = error_count / total_cells

    dominant_rows = sorted(range(len(row_error_counts)), key=lambda i: row_error_counts[i], reverse=True)[:3]
    dominant_cols = sorted(range(len(col_error_counts)), key=lambda i: col_error_counts[i], reverse=True)[:3]

    return {
        "same_shape": same_shape_flag,
        "pred_shape": (ph, pw),
        "target_shape": (th, tw),
        "error_map": error_map,
        "error_count": int(error_count),
        "error_density": _clip01(density),
        "pixel_accuracy": _clip01(pixel_accuracy),
        "color_score": _clip01(color_score),
        "structural_score": _clip01(structural_score),
        "row_error_counts": row_error_counts,
        "col_error_counts": col_error_counts,
        "dominant_rows": dominant_rows,
        "dominant_cols": dominant_cols,
        "bbox_pred": bbox_pred,
        "bbox_target": bbox_tgt,
        "bbox_mismatch": bbox_mismatch,
    }

def calculate_aofe_score(pred, target):
    a = analyze_errors(pred, target)
    score = (
        0.60 * a["pixel_accuracy"] +
        0.25 * a["structural_score"] +
        0.15 * a["color_score"]
    )
    return _clip01(score), a

# ============================================
# PROGRAM EVAL
# ============================================

def infer_failure_signature(error_analyses):
    if not error_analyses:
        return {
            "type": "no_data",
            "pattern": "no_train_pairs",
            "error_density_trend": [],
            "likely_missing_operator": None,
        }

    densities = [ea["error_density"] for ea in error_analyses]
    same_shape_ratio = sum(int(ea["same_shape"]) for ea in error_analyses) / len(error_analyses)
    bbox_mismatch_ratio = sum(int(ea["bbox_mismatch"]) for ea in error_analyses) / len(error_analyses)

    if same_shape_ratio < 0.5:
        ftype = "shape_mismatch"
        missing = "tile_or_crop_or_transform"
    elif bbox_mismatch_ratio > 0.5:
        ftype = "object_placement_mismatch"
        missing = "translate_or_object_operator"
    elif np.mean(densities) < 0.25:
        ftype = "near_miss_or_missing_operator"
        missing = "specialized_refinement_operator"
    else:
        ftype = "content_mismatch"
        missing = "color_or_structure_operator"

    pattern = (
        f"same_shape_ratio={same_shape_ratio:.2f}, "
        f"bbox_mismatch_ratio={bbox_mismatch_ratio:.2f}, "
        f"avg_density={np.mean(densities):.3f}"
    )

    return {
        "type": ftype,
        "pattern": pattern,
        "error_density_trend": [round(d, 4) for d in densities],
        "likely_missing_operator": missing,
    }

def eval_program_with_errors(program, task):
    program = _normalize_program(program)

    per_pair_scores = []
    error_analyses = []
    outputs = []

    for pair in task["train"]:
        pred = run_program_compat(program, pair["input"])
        tgt = _safe_to_list(pair["output"])
        score, analysis = calculate_aofe_score(pred, tgt)
        per_pair_scores.append(score)
        error_analyses.append(analysis)
        outputs.append(pred)

    avg_score = float(np.mean(per_pair_scores)) if per_pair_scores else 0.0
    exact = all(abs(s - 1.0) < 1e-12 for s in per_pair_scores) if per_pair_scores else False
    mdl = _mdl_cost(program)
    failure_signature = infer_failure_signature(error_analyses)

    result = {
        "program": program,
        "outputs": outputs,
        "pair_scores": per_pair_scores,
        "avg_score": avg_score,
        "exact": exact,
        "mdl": mdl,
        "error_analyses": error_analyses,
        "failure_signature": failure_signature,
    }
    return result

# ============================================
# TASK-AWARE OP BUILDING
# ============================================

def infer_color_mapping_from_task(task):
    maps = []
    for pair in task["train"]:
        inp = np.array(_safe_to_list(pair["input"]), dtype=int)
        out = np.array(_safe_to_list(pair["output"]), dtype=int)
        if inp.shape != out.shape:
            continue

        mapping = {}
        valid = True
        for src in np.unique(inp):
            vals = np.unique(out[inp == src])
            if len(vals) != 1:
                valid = False
                break
            mapping[int(src)] = int(vals[0])

        if valid and mapping:
            maps.append(mapping)

    if not maps:
        return None

    first = maps[0]
    for m in maps[1:]:
        if m != first:
            return None
    return first

def infer_scale_factor(task):
    factors = []
    for pair in task["train"]:
        ih, iw = _grid_shape(pair["input"])
        oh, ow = _grid_shape(pair["output"])
        if ih > 0 and iw > 0 and oh % ih == 0 and ow % iw == 0 and (oh // ih) == (ow // iw):
            factors.append(oh // ih)
        else:
            return None
    if not factors:
        return None
    if len(set(factors)) == 1:
        return factors[0]
    return None

def infer_row_alt_tiling(task, n):
    for pair in task["train"]:
        inp = np.array(_safe_to_list(pair["input"]), dtype=int)
        out = np.array(_safe_to_list(pair["output"]), dtype=int)
        ih, iw = inp.shape
        oh, ow = out.shape
        if oh != ih * n or ow != iw * n:
            return False

        blocks = []
        for i in range(n):
            row_blocks = []
            for _ in range(n):
                block = np.fliplr(inp) if (i % 2 == 1) else inp
                row_blocks.append(block)
            blocks.append(np.concatenate(row_blocks, axis=1))
        expected = np.concatenate(blocks, axis=0)
        if not np.array_equal(expected, out):
            return False
    return True

def build_task_ops(task):
    ops = [{"op": "identity", "params": {}}]

    # Base transforms
    for name in ["rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "transpose", "crop_to_bbox", "extract_largest_object"]:
        ops.append({"op": name, "params": {}})

    # Translation sweep
    for dx in [-2, -1, 0, 1, 2]:
        for dy in [-2, -1, 0, 1, 2]:
            if dx == 0 and dy == 0:
                continue
            ops.append({"op": "translate", "params": {"dx": dx, "dy": dy, "fill": 0}})

    # Scale-aware tiling
    n = infer_scale_factor(task)
    if n is not None and 1 <= n <= 4:
        ops.append({"op": "tile_n", "params": {"n": int(n)}})
        if infer_row_alt_tiling(task, int(n)):
            ops.append({"op": "row_alt_tile_n", "params": {"n": int(n)}})
    else:
        for cand_n in [2, 3]:
            ops.append({"op": "tile_n", "params": {"n": cand_n}})
            ops.append({"op": "row_alt_tile_n", "params": {"n": cand_n}})

    # Full color map if consistent
    cmap = infer_color_mapping_from_task(task)
    if cmap is not None:
        ops.append({"op": "color_map", "params": {"mapping": cmap}})

    # Conditional recolors from observed colors
    train_colors_in = sorted(set(v for pair in task["train"] for row in pair["input"] for v in row))
    train_colors_out = sorted(set(v for pair in task["train"] for row in pair["output"] for v in row))
    for src in train_colors_in:
        for dst in train_colors_out:
            if src != dst:
                ops.append({"op": "conditional_color_map", "params": {"source": int(src), "target": int(dst)}})

    # Deduplicate
    seen = set()
    deduped = []
    for op in ops:
        key = (op["op"], _make_hashable(op["params"]))
        if key not in seen:
            deduped.append(op)
            seen.add(key)

    return deduped

# ============================================
# PRIORITY OPS
# ============================================

def get_priority_ops(eval_result, task):
    failure = eval_result["failure_signature"]["type"]
    pair_errors = eval_result["error_analyses"]

    priorities = []

    # Task-aware defaults
    task_ops = build_task_ops(task)

    if failure == "shape_mismatch":
        preferred = {"tile_n", "row_alt_tile_n", "crop_to_bbox", "rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "transpose", "extract_largest_object"}
    elif failure == "object_placement_mismatch":
        preferred = {"translate", "rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v", "transpose"}
    elif failure == "near_miss_or_missing_operator":
        preferred = {"conditional_color_map", "color_map", "translate", "crop_to_bbox", "extract_largest_object"}
    else:
        preferred = {"conditional_color_map", "color_map", "translate", "rotate90", "reflect_h"}

    # Additional signal from density concentration
    dominant_rows = []
    dominant_cols = []
    for ea in pair_errors:
        dominant_rows.extend(ea["dominant_rows"])
        dominant_cols.extend(ea["dominant_cols"])

    row_concentrated = len(dominant_rows) > 0 and len(set(dominant_rows[:3])) <= 3
    col_concentrated = len(dominant_cols) > 0 and len(set(dominant_cols[:3])) <= 3

    if row_concentrated or col_concentrated:
        preferred = preferred | {"translate", "reflect_h", "reflect_v"}

    for op in task_ops:
        score = 0
        if op["op"] in preferred:
            score += 10

        if op["op"] == "translate":
            dx = abs(int(op["params"].get("dx", 0)))
            dy = abs(int(op["params"].get("dy", 0)))
            score += max(0, 4 - dx - dy)

        if op["op"] in {"conditional_color_map", "color_map"}:
            score += 3

        if op["op"] == "identity":
            score -= 100

        priorities.append((score, op))

    priorities.sort(key=lambda x: x[0], reverse=True)
    return [op for _, op in priorities]

# ============================================
# SEED GENERATION
# ============================================

def generate_seed_programs(task):
    seeds = [
        [{"op": "identity", "params": {}}],
        [{"op": "rotate90", "params": {}}],
        [{"op": "rotate180", "params": {}}],
        [{"op": "rotate270", "params": {}}],
        [{"op": "reflect_h", "params": {}}],
        [{"op": "reflect_v", "params": {}}],
        [{"op": "transpose", "params": {}}],
        [{"op": "crop_to_bbox", "params": {}}],
        [{"op": "extract_largest_object", "params": {}}],
    ]

    n = infer_scale_factor(task)
    if n is not None and 1 <= n <= 4:
        seeds.append([{"op": "tile_n", "params": {"n": int(n)}}])
        if infer_row_alt_tiling(task, int(n)):
            seeds.append([{"op": "row_alt_tile_n", "params": {"n": int(n)}}])

    cmap = infer_color_mapping_from_task(task)
    if cmap is not None:
        seeds.append([{"op": "color_map", "params": {"mapping": cmap}}])

    return [_normalize_program(s) for s in seeds]

# ============================================
# PROGRAM EXPANSION
# ============================================

def expand_program(program, priority_ops, max_depth=3):
    program = _normalize_program(program)
    children = []

    if len(program) >= max_depth:
        return children

    for op in priority_ops[:12]:
        child = deepcopy(program) + [deepcopy(op)]
        children.append(child)

    return children

# ============================================
# BEAM SEARCH OVERRIDE
# ============================================

def beam_search(task, beam_width=24, max_iters=6, max_depth=3, verbose=True):
    visited = set()
    beam = []
    all_evals = []

    seed_programs = generate_seed_programs(task)

    for prog in seed_programs:
        key = _program_to_key(prog)
        if key in visited:
            continue
        visited.add(key)
        ev = eval_program_with_errors(prog, task)
        beam.append(ev)
        all_evals.append(ev)

    beam.sort(key=lambda e: (e["avg_score"], -e["mdl"]), reverse=True)
    beam = beam[:beam_width]

    if verbose:
        print("=" * 60)
        print("AOFE beam_search override active")
        print("Initial seeds:", len(seed_programs))
        print("Initial beam size:", len(beam))
        print("=" * 60)

    best = beam[0] if beam else None
    refinement_trace = []

    for step_idx in range(max_iters):
        if not beam:
            break

        beam.sort(key=lambda e: (e["avg_score"], -e["mdl"]), reverse=True)
        best = beam[0]

        density_progress = [
            round(np.mean([ea["error_density"] for ea in cand["error_analyses"]]), 4)
            for cand in beam[:5]
        ]
        refinement_trace.append({
            "iter": step_idx,
            "best_score": round(best["avg_score"], 6),
            "best_mdl": best["mdl"],
            "best_program": best["program"],
            "top5_density": density_progress,
        })

        if verbose:
            print(f"[ITER {step_idx}] best_score={best['avg_score']:.6f} mdl={best['mdl']} failure={best['failure_signature']['type']}")
            print(" best_program =", best["program"])
            print(" top5_density =", density_progress)

        if best["exact"]:
            if verbose:
                print("Exact solution found.")
            break

        new_candidates = []

        for cand in beam:
            priority_ops = get_priority_ops(cand, task)
            children = expand_program(cand["program"], priority_ops, max_depth=max_depth)

            for child in children:
                key = _program_to_key(child)
                if key in visited:
                    continue
                visited.add(key)
                ev = eval_program_with_errors(child, task)
                new_candidates.append(ev)
                all_evals.append(ev)

        merged = beam + new_candidates
        merged.sort(key=lambda e: (e["avg_score"], -e["mdl"]), reverse=True)
        beam = merged[:beam_width]

    best = sorted(all_evals, key=lambda e: (e["avg_score"], -e["mdl"]), reverse=True)[0] if all_evals else None

    if best:
        for step in best["program"]:
            OP_USAGE[step["op"]] += 1

    return {
        "best_program": best["program"] if best else [],
        "best_score": best["avg_score"] if best else 0.0,
        "best_eval": best,
        "refinement_trace": refinement_trace,
        "beam_size": beam_width,
        "visited_count": len(visited),
    }

# ============================================
# INTROSPECTION
# ============================================

def _safe_source_preview(fn, n=300):
    try:
        src = inspect.getsource(fn)
        return src[:n]
    except Exception:
        return "<source unavailable>"

def introspect_system(task=None):
    print("\n" + "=" * 70)
    print("AOFE INTROSPECTION")
    print("=" * 70)

    print("CELL_REGISTRY:", CELL_REGISTRY)
    print("SYSTEM_STATE:", SYSTEM_STATE)
    print("FILES_INDEX count:", len(FILES_INDEX))

    print("\n[beam_search source preview]")
    print(_safe_source_preview(beam_search, 300))

    print("\n[build_task_ops source preview]")
    print(_safe_source_preview(build_task_ops, 300))

    overrides = [
        "analyze_errors",
        "calculate_aofe_score",
        "eval_program_with_errors",
        "get_priority_ops",
        "build_task_ops",
        "beam_search",
    ]
    print("\nActive overrides:", overrides)

    if task is not None:
        ops = build_task_ops(task)
        op_counts = defaultdict(int)
        for op in ops:
            op_counts[op["op"]] += 1
        print("\nOperator count:", len(ops))
        print("Operator distribution:", dict(sorted(op_counts.items())))

    print("=" * 70 + "\n")

# ============================================
# CONTROLLED TESTS
# ============================================

def make_controlled_tasks():
    # 1) simple recolor
    recolor_task = {
        "train": [
            {
                "input": [[1, 0], [0, 1]],
                "output": [[2, 0], [0, 2]],
            },
            {
                "input": [[1, 1], [0, 0]],
                "output": [[2, 2], [0, 0]],
            },
        ],
        "test": [{"input": [[0, 1], [1, 0]]}],
    }

    # 2) rotation
    rotation_task = {
        "train": [
            {
                "input": [[1, 0, 0], [1, 1, 0]],
                "output": _safe_to_list(np.rot90(np.array([[1, 0, 0], [1, 1, 0]]), 1)),
            },
            {
                "input": [[0, 2, 0], [2, 2, 2]],
                "output": _safe_to_list(np.rot90(np.array([[0, 2, 0], [2, 2, 2]]), 1)),
            },
        ],
        "test": [{"input": [[3, 0, 0], [3, 3, 0]]}],
    }

    # 3) row-alt tiling
    object_task = {
        "train": [
            {
                "input": [[7, 9], [4, 3]],
                "output": [
                    [7, 9, 7, 9, 7, 9],
                    [4, 3, 4, 3, 4, 3],
                    [9, 7, 9, 7, 9, 7],
                    [3, 4, 3, 4, 3, 4],
                    [7, 9, 7, 9, 7, 9],
                    [4, 3, 4, 3, 4, 3],
                ],
            },
            {
                "input": [[8, 6], [6, 4]],
                "output": [
                    [8, 6, 8, 6, 8, 6],
                    [6, 4, 6, 4, 6, 4],
                    [6, 8, 6, 8, 6, 8],
                    [4, 6, 4, 6, 4, 6],
                    [8, 6, 8, 6, 8, 6],
                    [6, 4, 6, 4, 6, 4],
                ],
            },
        ],
        "test": [{"input": [[3, 2], [7, 8]]}],
    }

    return {
        "simple_recolor": recolor_task,
        "rotation": rotation_task,
        "object_tiling": object_task,
    }

def run_controlled_tests(beam_width=24, max_iters=6, max_depth=3):
    global RUN_REPORT, PAST_RUNS

    tasks = make_controlled_tasks()
    results = {}
    solved = 0
    scores = []

    for task_name, task in tasks.items():
        print("\n" + "#" * 70)
        print("RUNNING CONTROLLED TASK:", task_name)
        print("#" * 70)

        introspect_system(task)

        result = beam_search(
            task,
            beam_width=beam_width,
            max_iters=max_iters,
            max_depth=max_depth,
            verbose=True
        )

        best_eval = result["best_eval"]
        best_score = float(result["best_score"])
        scores.append(best_score)
        solved += int(best_eval is not None and best_eval["exact"])

        operators_used = [step["op"] for step in best_eval["program"]] if best_eval else []

        density_prog = [
            round(float(np.mean(step["top5_density"])), 4) if step["top5_density"] else 1.0
            for step in result["refinement_trace"]
        ]

        results[task_name] = {
            "score": best_score,
            "program": best_eval["program"] if best_eval else [],
            "failure_mode": best_eval["failure_signature"]["type"] if best_eval else "no_result",
            "operators_used": operators_used,
            "refinement_steps": len(result["refinement_trace"]),
            "error_density_progression": density_prog,
        }

        print("\nBEST PROGRAM:", results[task_name]["program"])
        print("BEST SCORE:", results[task_name]["score"])
        print("REFINEMENT STEPS:", results[task_name]["refinement_steps"])
        print("FAILURE MODE:", results[task_name]["failure_mode"])
        print("OPERATORS USED:", results[task_name]["operators_used"])
        print("ERROR DENSITY PROGRESSION:", results[task_name]["error_density_progression"])

    avg_score = float(np.mean(scores)) if scores else 0.0
    solve_rate = float(solved / max(1, len(tasks)))

    RUN_REPORT = {
        "avg_score": avg_score,
        "solve_rate": solve_rate,
        "task_results": results,
    }

    SYSTEM_STATE["beam_search_version"] = "aofe_v13_5_density_guided_override"
    SYSTEM_STATE["ops_version"] = "task_aware_ops_v1"
    SYSTEM_STATE["priority_version"] = "density_guided_priority_v1"
    SYSTEM_STATE["last_run_summary"] = {
        "avg_score": avg_score,
        "solve_rate": solve_rate,
        "num_tasks": len(tasks),
    }

    unused_ops = []
    task_ops_union = set()
    for task in tasks.values():
        for op in build_task_ops(task):
            task_ops_union.add(op["op"])

    for op_name in sorted(task_ops_union):
        if OP_USAGE[op_name] == 0:
            unused_ops.append(op_name)

    high_impact_ops = sorted(OP_USAGE.items(), key=lambda x: x[1], reverse=True)

    run_memory_item = {
        "config": {
            "beam_width": beam_width,
            "max_iters": max_iters,
            "max_depth": max_depth,
        },
        "avg_score": avg_score,
        "notes": {
            "solve_rate": solve_rate,
            "unused_operators": unused_ops,
            "high_impact_operators": high_impact_ops[:10],
        }
    }
    PAST_RUNS.append(run_memory_item)

    print("\n" + "=" * 70)
    print("FINAL RUN REPORT")
    print("=" * 70)
    print(RUN_REPORT)
    print("\nUNUSED OPERATORS:", unused_ops)
    print("HIGH-IMPACT OPERATORS:", high_impact_ops[:10])
    print("PAST_RUNS length:", len(PAST_RUNS))
    print("SYSTEM_STATE updated:", SYSTEM_STATE)
    print("=" * 70)

    return RUN_REPORT

print("AOFE patch cell loaded successfully.")

# ============================================
# OPTIONAL AUTO-RUN
# ============================================

CONTROLLED_REPORT = run_controlled_tests(
    beam_width=24,
    max_iters=6,
    max_depth=3
)

AOFE patch cell starting...
FILES_INDEX entries: 10
AOFE patch cell loaded successfully.

######################################################################
RUNNING CONTROLLED TASK: simple_recolor
######################################################################

AOFE INTROSPECTION
CELL_REGISTRY: {'07': 'operator_system', '10': 'priority_system', '11c': 'beam_core', '11d': 'seed_templates', '12': 'aofe_v13_5_core_patch'}
SYSTEM_STATE: {'active_cells': ['CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH', 'CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH'], 'beam_search_version': 'aofe_v13_5_density_guided_override', 'ops_version': 'task_aware_ops_v1', 'priority_version': 'structure_boost_v2', 'last_run_summary': {'avg_score': 1.0, 'solve_rate': 1.0, 'num_tasks': 3}}
FILES_INDEX count: 10

[beam_search source preview]
def beam_search(task, beam_width=24, max_iters=6, max_depth=3, verbose=True):
    visited = set()
    beam = []
    all_evals = []

    seed_programs = generate_seed_program

In [45]:
# ============================================
# CELL 12b — RUN CONTROLLED TASK (COMPATIBLE)
# ============================================

def run_controlled_task(
    name,
    task,
    width=None,
    depth=None,
    beam_width=None,
    max_iters=None,
    max_depth=None
):
    """
    Backward + forward compatible runner
    Supports BOTH:
        width/depth  (old)
        beam_width/max_iters/max_depth (new)
    """

    # ----------------------------------
    # PARAM RESOLUTION
    # ----------------------------------

    # map old → new if needed
    if beam_width is None:
        beam_width = width if width is not None else 24

    if max_depth is None:
        max_depth = depth if depth is not None else 3

    if max_iters is None:
        max_iters = 6

    print("=" * 70)
    print(f"TEST: {name}")
    print("=" * 70)

    result = beam_search(
        task,
        beam_width=beam_width,
        max_iters=max_iters,
        max_depth=max_depth,
        verbose=True
    )

    best = result["best_eval"]

    print("\nBest program:")
    print(result["best_program"])
    print(f"Score: {result['best_score']:.4f}")
    print(f"Refinement steps: {len(result['refinement_trace'])}")

    if best:
        print(f"Failure mode: {best['failure_signature']['type']}")

        densities = [
            round(float(np.mean(step["top5_density"])), 4)
            for step in result["refinement_trace"]
            if step["top5_density"]
        ]

        print(f"Error density progression: {densities}")

    return {
        "best_program": result["best_program"],
        "best_score": result["best_score"],
        "refinement_steps": len(result["refinement_trace"]),
        "failure_mode": best["failure_signature"]["type"] if best else "no_result",
        "error_density_progression": densities if best else [],
    }

print("CELL 12b loaded (compatible mode).")

CELL 12b loaded (compatible mode).


In [46]:
# ============================================
# CELL 13 — PERFORMANCE SUMMARY METRICS
# ============================================

def summarize_results(results):
    scores = []
    solves = 0

    for task_id, res in results:
        score = res["best_score"]
        scores.append(score)

        if score >= 0.999:
            solves += 1

    avg_score = sum(scores) / len(scores) if scores else 0
    solve_rate = solves / len(scores) if scores else 0

    print("\n" + "=" * 70)
    print("PERFORMANCE SUMMARY")
    print("=" * 70)
    print(f"Tasks evaluated: {len(scores)}")
    print(f"Average score: {avg_score:.4f}")
    print(f"Solve rate: {solve_rate:.2%}")
    print("=" * 70)

    return {
        "avg_score": avg_score,
        "solve_rate": solve_rate,
        "total_tasks": len(scores)
    }

print("CELL 13 loaded.")

CELL 13 loaded.


In [47]:
# ============================================
# CELL 14 — CONNECT GOOGLE DRIVE
# ============================================

from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [48]:
# ============================================
# CELL 15 — LOGGING SYSTEM SETUP
# ============================================

import os
import json
from datetime import datetime

LOG_DIR = "/content/drive/MyDrive/AOFE_LOGS"
LOG_FILE = os.path.join(LOG_DIR, "aofe_progress_log.json")

# create directory if needed
os.makedirs(LOG_DIR, exist_ok=True)

# initialize file if not exists
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        json.dump([], f)

print("Log system ready.")
print("Log file:", LOG_FILE)

Log system ready.
Log file: /content/drive/MyDrive/AOFE_LOGS/aofe_progress_log.json


In [49]:
# ============================================
# CELL 16 — LOG RESULTS FUNCTION
# ============================================

def log_run(results, config):

    with open(LOG_FILE, "r") as f:
        logs = json.load(f)

    entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "config": config,
        "summary": [],
    }

    for task_id, res in results:
        entry["summary"].append({
            "task_id": task_id,
            "score": res["best_score"],
            "steps": res["refinement_steps"],
            "failure": res["failure_mode"],
            "program_length": len(res["best_program"]) if res["best_program"] else 0
        })

    logs.append(entry)

    with open(LOG_FILE, "w") as f:
        json.dump(logs, f, indent=2)

    print("✅ Run logged successfully.")

In [50]:
# ============================================
# CELL 17 — BENCHMARK (DEEP SEARCH MODE)
# ============================================

results = []

for i, (task_id, task) in enumerate(solver_tasks.items()):
    if i >= 5:
        break

    print("\n" + "=" * 70)
    print(f"TASK {i} | ID: {task_id}")
    print("=" * 70)

    result = run_controlled_task(
        f"ARC Task {task_id}",
        task,
        width=30,     # 🔥 increased
        depth=6       # 🔥 increased
    )

    results.append((task_id, result))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

for task_id, res in results:
    print(f"\nTask: {task_id}")
    print(f"  Score: {res['best_score']:.4f}")
    print(f"  Steps: {res['refinement_steps']}")
    print(f"  Failure: {res['failure_mode']}")
    print(f"  Program: {res['best_program']}")

# ----------------------------------
# METRICS
# ----------------------------------
summary_stats = summarize_results(results)

# ----------------------------------
# LOGGING
# ----------------------------------
config = {
    "width": 30,
    "depth": 6,
    "mode": "deep_search_unlock",
    "notes": "enable finalization operators"
}

log_run(results, config)


TASK 0 | ID: 00576224
TEST: ARC Task 00576224
AOFE beam_search override active
Initial seeds: 11
Initial beam size: 11
[ITER 0] best_score=1.000000 mdl=2 failure=near_miss_or_missing_operator
 best_program = [{'op': 'row_alt_tile_n', 'params': {'n': 3}}]
 top5_density = [np.float64(0.0), np.float64(0.3333), np.float64(0.8889), np.float64(0.8889), np.float64(0.9167)]
Exact solution found.

Best program:
[{'op': 'row_alt_tile_n', 'params': {'n': 3}}]
Score: 1.0000
Refinement steps: 1
Failure mode: near_miss_or_missing_operator
Error density progression: [0.6056]

TASK 1 | ID: 007bbfb7
TEST: ARC Task 007bbfb7
AOFE beam_search override active
Initial seeds: 10
Initial beam size: 10
[ITER 0] best_score=0.830060 mdl=2 failure=near_miss_or_missing_operator
 best_program = [{'op': 'tile_n', 'params': {'n': 3}}]
 top5_density = [np.float64(0.2222), np.float64(0.9136), np.float64(0.921), np.float64(0.9333), np.float64(0.9383)]
[ITER 1] best_score=0.830060 mdl=2 failure=near_miss_or_missing_oper

In [51]:
print("numpy:", "OK" if "np" in globals() else "MISSING")
print("extract_objects:", "OK" if "extract_objects" in globals() else "MISSING")
print("run_program:", "OK" if "run_program" in globals() else "NOT FOUND (safe)")
print("beam_search:", "OK" if "beam_search" in globals() else "WILL BE CREATED")

numpy: OK
extract_objects: OK
run_program: OK
beam_search: OK


In [52]:
# ============================================
# CELL 18 — SYSTEM INTROSPECTION (FULL TRACE)
# ============================================

import inspect

def _safe_preview(fn, lines=20):
    try:
        src = inspect.getsource(fn).split("\n")
        return "\n".join(src[:lines])
    except Exception:
        return "<source unavailable>"

def _print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def _print_kv(key, value):
    print(f"{key:<25}: {value}")

def trace_system():

    _print_section("SYSTEM STATE")

    _print_kv("Active Cells", SYSTEM_STATE.get("active_cells"))
    _print_kv("Beam Version", SYSTEM_STATE.get("beam_search_version"))
    _print_kv("Ops Version", SYSTEM_STATE.get("ops_version"))
    _print_kv("Priority Version", SYSTEM_STATE.get("priority_version"))
    _print_kv("Last Run", SYSTEM_STATE.get("last_run_summary"))

    _print_section("CELL REGISTRY")

    for k, v in sorted(CELL_REGISTRY.items()):
        print(f"CELL {k:<5} → {v}")

    _print_section("CORE FUNCTION TRACE")

    functions = [
        "run_program",
        "run_program_compat",
        "execute_step",
        "analyze_errors",
        "calculate_aofe_score",
        "eval_program_with_errors",
        "build_task_ops",
        "get_priority_ops",
        "generate_seed_programs",
        "expand_program",
        "beam_search",
    ]

    for fn_name in functions:
        print("\n" + "-" * 80)
        print(f"FUNCTION: {fn_name}")

        if fn_name in globals():
            fn = globals()[fn_name]
            print("STATUS: FOUND")

            # identify likely owner cell
            owner = None
            for cell_id, name in CELL_REGISTRY.items():
                if name.lower() in fn_name.lower():
                    owner = cell_id

            print(f"LIKELY CELL: {owner if owner else 'UNKNOWN'}")

            print("\nSOURCE PREVIEW:")
            print(_safe_preview(fn, lines=25))
        else:
            print("STATUS: MISSING")

    _print_section("OPERATOR SNAPSHOT")

    try:
        sample_task = {
            "train": [
                {"input": [[1,0],[0,1]], "output": [[2,0],[0,2]]}
            ]
        }
        ops = build_task_ops(sample_task)

        print("Total Operators:", len(ops))

        counts = {}
        for op in ops:
            counts[op["op"]] = counts.get(op["op"], 0) + 1

        print("\nOperator Distribution:")
        for k, v in sorted(counts.items()):
            print(f"  {k:<25} : {v}")

    except Exception as e:
        print("Operator build failed:", e)

    _print_section("USAGE + MEMORY")

    print("OP_USAGE (top 10):")
    top_ops = sorted(OP_USAGE.items(), key=lambda x: x[1], reverse=True)[:10]
    for op, count in top_ops:
        print(f"  {op:<25} : {count}")

    print("\nPAST_RUNS count:", len(PAST_RUNS))

    _print_section("BEAM SEARCH SNAPSHOT")

    if "beam_search" in globals():
        print(_safe_preview(beam_search, 40))
    else:
        print("beam_search NOT FOUND")

    print("\n" + "=" * 80)
    print("END OF TRACE")
    print("=" * 80)


# ============================================
# RUN TRACE
# ============================================

trace_system()


SYSTEM STATE
Active Cells             : ['CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH', 'CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH']
Beam Version             : aofe_v13_5_density_guided_override
Ops Version              : task_aware_ops_v1
Priority Version         : density_guided_priority_v1
Last Run                 : {'avg_score': 1.0, 'solve_rate': 1.0, 'num_tasks': 3}

CELL REGISTRY
CELL 07    → operator_system
CELL 10    → priority_system
CELL 11c   → beam_core
CELL 11d   → seed_templates
CELL 12    → aofe_v13_5_core_patch

CORE FUNCTION TRACE

--------------------------------------------------------------------------------
FUNCTION: run_program
STATUS: FOUND
LIKELY CELL: UNKNOWN

SOURCE PREVIEW:
def run_program(program, grid):

    current = to_np(grid)

    for op_name, params in program:

        try:
            # ----------------------------------
            # TRANSFORM OPS
            # ----------------------------------
            if op_name == "rotate90":
            

In [53]:
# ============================================
# CELL 18b — REGISTRY SANITIZER
# ============================================

def clean_registry():

    global CELL_REGISTRY, SYSTEM_STATE

    print("\nCleaning CELL REGISTRY...")

    # remove legacy XX entry
    if "XX" in CELL_REGISTRY:
        del CELL_REGISTRY["XX"]
        print("Removed CELL XX")

    # clean active cells
    SYSTEM_STATE["active_cells"] = [
        c for c in SYSTEM_STATE["active_cells"]
        if "XX" not in c
    ]

    # enforce correct mapping
    CELL_REGISTRY["12"] = "aofe_v13_5_core_patch"

    print("\nCleaned CELL_REGISTRY:")
    for k, v in sorted(CELL_REGISTRY.items()):
        print(f"CELL {k} → {v}")

    print("\nActive Cells:")
    print(SYSTEM_STATE["active_cells"])

    print("\nRegistry clean complete.")

clean_registry()


Cleaning CELL REGISTRY...

Cleaned CELL_REGISTRY:
CELL 07 → operator_system
CELL 10 → priority_system
CELL 11c → beam_core
CELL 11d → seed_templates
CELL 12 → aofe_v13_5_core_patch

Active Cells:
['CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH', 'CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH']

Registry clean complete.


In [54]:
clean_registry()


Cleaning CELL REGISTRY...

Cleaned CELL_REGISTRY:
CELL 07 → operator_system
CELL 10 → priority_system
CELL 11c → beam_core
CELL 11d → seed_templates
CELL 12 → aofe_v13_5_core_patch

Active Cells:
['CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH', 'CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH']

Registry clean complete.


In [55]:
trace_system()


SYSTEM STATE
Active Cells             : ['CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH', 'CELL 12 — AOFE v13.5 CORE INTEGRATION PATCH']
Beam Version             : aofe_v13_5_density_guided_override
Ops Version              : task_aware_ops_v1
Priority Version         : density_guided_priority_v1
Last Run                 : {'avg_score': 1.0, 'solve_rate': 1.0, 'num_tasks': 3}

CELL REGISTRY
CELL 07    → operator_system
CELL 10    → priority_system
CELL 11c   → beam_core
CELL 11d   → seed_templates
CELL 12    → aofe_v13_5_core_patch

CORE FUNCTION TRACE

--------------------------------------------------------------------------------
FUNCTION: run_program
STATUS: FOUND
LIKELY CELL: UNKNOWN

SOURCE PREVIEW:
def run_program(program, grid):

    current = to_np(grid)

    for op_name, params in program:

        try:
            # ----------------------------------
            # TRANSFORM OPS
            # ----------------------------------
            if op_name == "rotate90":
            